# Included Johnnie Walker Black Label stoppage events

This notebook:

1. Loads the PLC stoppage-event report.
2. Uses `Orders.xlsx` to map order numbers to products.
3. Keeps events whose status is `Include`.
4. Keeps orders mapped to `JW Black` (Johnnie Walker Black Label).
5. Returns the filtered DataFrame as `johnnie_walker_black_label_events`.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display


# Find the data folder whether Jupyter starts here, at the repository root,
# or from the parent project directory.
working_directory = Path.cwd().resolve()
search_roots = [working_directory, *working_directory.parents]
candidate_directories = []
for search_root in search_roots:
    candidate_directories.extend([
        search_root,
        search_root / "data" / "plc_l14_events",
        search_root / "stoppage_detection_and_classification" / "plc_stoppage_events" / "data" / "plc_l14_events",
        search_root / "24H_Insights" / "stoppage_detection_and_classification" / "plc_stoppage_events" / "data" / "plc_l14_events",
    ])

data_directory = next(
    candidate
    for candidate in candidate_directories
    if (candidate / "L14 Shrinkwrap - Unhealth Events FYTD.xls").is_file()
    and (candidate / "Orders.xlsx").is_file()
)

stoppage_events_path = data_directory / "L14 Shrinkwrap - Unhealth Events FYTD.xls"
orders_path = data_directory / "Orders.xlsx"

print(f"Stoppage events: {stoppage_events_path}")
print(f"Order mapping:   {orders_path}")


In [ ]:
# The stoppage workbook has three report-title rows above its real header.
plc_stoppage_events = pd.read_excel(
    stoppage_events_path,
    sheet_name="Machine Health and Stops Report",
    header=3,
)
orders = pd.read_excel(
    orders_path,
    sheet_name="Sheet1",
)

# Replace embedded line breaks and surrounding whitespace in column names.
plc_stoppage_events.columns = [
    " ".join(str(column_name).split())
    for column_name in plc_stoppage_events.columns
]
orders.columns = [
    " ".join(str(column_name).split())
    for column_name in orders.columns
]

# Identify the Include/Exclude column by its values because its Excel header is blank.
status_columns = []
for column_name in plc_stoppage_events.columns:
    normalized_values = set(
        plc_stoppage_events[column_name]
        .dropna()
        .astype(str)
        .str.strip()
        .str.casefold()
        .unique()
    )
    if normalized_values and normalized_values <= {"include", "exclude"}:
        status_columns.append(column_name)

if len(status_columns) != 1:
    raise ValueError(
        "Expected exactly one Include/Exclude column, "
        f"found: {status_columns}"
    )
plc_stoppage_events = plc_stoppage_events.rename(
    columns={status_columns[0]: "Include_Status"}
)

# Validate the fields needed for filtering and mapping.
required_event_columns = {"Order", "Include_Status"}
required_order_columns = {"Order", "Description", "Product"}
missing_event_columns = required_event_columns - set(plc_stoppage_events.columns)
missing_order_columns = required_order_columns - set(orders.columns)
if missing_event_columns:
    raise ValueError(f"Missing stoppage columns: {sorted(missing_event_columns)}")
if missing_order_columns:
    raise ValueError(f"Missing order columns: {sorted(missing_order_columns)}")

# Normalize both order keys to the same nullable integer type.
plc_stoppage_events["Order"] = pd.to_numeric(
    plc_stoppage_events["Order"],
    errors="coerce",
).astype("Int64")
orders["Order"] = pd.to_numeric(
    orders["Order"],
    errors="raise",
).astype("Int64")

# One order number must map to only one product.
if orders["Order"].duplicated().any():
    duplicate_orders = orders.loc[
        orders["Order"].duplicated(keep=False),
        "Order",
    ].tolist()
    raise ValueError(f"Duplicate order mappings found: {duplicate_orders}")

print(f"PLC stoppage rows loaded: {len(plc_stoppage_events):,}")
print(f"Order mappings loaded:    {len(orders):,}")


In [ ]:
# Keep the order mappings for Johnnie Walker Black Label.
jw_black_order_mapping = orders[
    orders["Product"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .eq("jw black")
].copy()

# Keep only stoppage rows explicitly marked Include.
included_stoppage_events = plc_stoppage_events[
    plc_stoppage_events["Include_Status"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .eq("include")
].copy()

# Join through the order mapping to identify the required product events.
johnnie_walker_black_label_events = included_stoppage_events.merge(
    jw_black_order_mapping[["Order", "Description", "Product"]],
    on="Order",
    how="inner",
    validate="many_to_one",
).reset_index(drop=True)

# Fail loudly if either filter has not been applied correctly.
if not johnnie_walker_black_label_events["Include_Status"].eq("Include").all():
    raise ValueError("The result contains a row not marked Include")
if not (
    johnnie_walker_black_label_events["Product"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .eq("jw black")
    .all()
):
    raise ValueError("The result contains a non-JW Black product")

print(f"JW Black order mappings: {len(jw_black_order_mapping):,}")
print(f"Included stoppage rows:  {len(included_stoppage_events):,}")
print(f"Returned events:         {len(johnnie_walker_black_label_events):,}")

# Save the dated extract under this notebook's output folder.
extract_date = pd.Timestamp.now(tz="Europe/London").strftime("%Y-%m-%d")
output_directory = data_directory.parent.parent / "output" / "01_prepare_events"
output_directory.mkdir(parents=True, exist_ok=True)
csv_output_path = (
    output_directory
    / f"johnnie_walker_black_label_events_{extract_date}.csv"
)
johnnie_walker_black_label_events.to_csv(csv_output_path, index=False)
print(f"Saved CSV extract:       {csv_output_path}")

# Return and display the requested DataFrame.
johnnie_walker_black_label_events
